In [ ]:
import pandas as pd
import numpy as np

## Endpoint data

In [ ]:
tests_df = pd.read_csv("../Data/tests.txt", sep="|", dtype=str, encoding="Windows-1252")
print(tests_df.shape)

tests_df = tests_df.replace(np.nan,'',regex=True)
print(tests_df.shape)


tests_df.head()

Filtering using the aquachem chemicals' CASRNs

In [ ]:
list(tests_df.columns)

In [ ]:
aquachem_df = pd.read_csv('../../Main_data/Data/ALL_aquaculture_chemicals.tsv',dtype=str,sep='\t')
aquachem_df = aquachem_df.replace(np.nan, '', regex=True)
print(aquachem_df.shape)
aquachem_df.head()

In [ ]:
aquachem_df['PubChemID'] = aquachem_df['PubChemID'].str.replace('CID:', '').str.strip()
aquachem_df['CASRN'] = aquachem_df['CASRN'].str.replace('CAS:', '').str.strip()
print(aquachem_df.shape)
aquachem_df.head()

In [ ]:
aquachems_cas = set(aquachem_df['CASRN']) - {''}
len(aquachems_cas)

In [ ]:
aquachem_df['cas_stripped'] = aquachem_df['CASRN'].str.replace('-', '')
print(aquachem_df.shape)
aquachem_df.head()

In [ ]:
aquachems = set(aquachem_df['cas_stripped']) - {''}
len(aquachems)

In [ ]:
aquachems_overlap = set(aquachems).intersection(set(tests_df['test_cas']))
print(len(aquachems_overlap))

In [ ]:
tests_df.columns

In [ ]:
tests_aquachem_df = pd.DataFrame(tests_df[tests_df['test_cas'].isin(aquachems_overlap)]).reset_index(drop=True)
print(tests_aquachem_df.shape)
tests_aquachem_df.head()

Results data

In [ ]:
results_df = pd.read_csv("../Data/results.txt", sep="|", dtype=str, encoding="Windows-1252")
print(results_df.shape)
results_df.head()

In [ ]:
list(results_df.columns)

In [ ]:
results_filtered_df = pd.DataFrame(results_df)
print(results_filtered_df.shape)
results_filtered_df = results_filtered_df.replace(np.nan,'',regex=True)
print(results_filtered_df.shape)
results_filtered_df.head()

In [ ]:
len(set(results_filtered_df['test_id']))

In [ ]:
results_filtered_df[results_filtered_df['test_id'].duplicated(keep=False)].sort_values(by='test_id')

Note: Same tests have resulted in different result giving rise to different result ids

In [ ]:
results_filtered_df[results_filtered_df['test_id'].isin(tests_aquachem_df['test_id'])].shape

Merging TESTS and RESULTS tables

In [ ]:
tests_results_df = results_filtered_df.merge(tests_aquachem_df, on="test_id", how="inner")
print(tests_results_df.shape)
tests_results_df.head()

Importing Chemicals table

In [ ]:
chemicals_df = pd.read_csv("../Data/validation/chemicals.txt", sep="|", dtype=str, encoding="Windows-1252")
print(chemicals_df.shape)
chemicals_df.head()

In [ ]:
tests_results_chems_df = tests_results_df.merge(chemicals_df, left_on="test_cas", right_on="cas_number", how="inner")
print(tests_results_chems_df.shape)
tests_results_chems_df.head()

Importing Species table

In [ ]:
species_df = pd.read_csv("../Data/validation/species.txt", sep="|", dtype=str, encoding="Windows-1252")
print(species_df.shape)
species_df.head()

In [ ]:
tests_results_chems_species_df = tests_results_chems_df.merge(species_df, on="species_number", how="inner")
print(tests_results_chems_species_df.shape)
tests_results_chems_species_df.head()

Importing Measurement table

Import MEASUREMENT, TREND, EFFECT, and REFERENCE tables

In [ ]:
measurement_codes = pd.read_csv("../Data/validation/measurement_codes.txt", sep="|", encoding='Windows-1252', dtype=str)
print(measurement_codes.shape)
measurement_codes.head()

In [ ]:
measurement_codes.columns = ["code", "measurement_description"]
print(measurement_codes.shape)
measurement_codes.head()

In [ ]:
'A1AR' in set(measurement_codes['code'])

In [ ]:
print(len(set(measurement_codes['code'])))

Checks

In [ ]:
print(len(set(tests_results_chems_species_df['measurement'])))

print(len(set(tests_results_chems_species_df['measurement']) - set(measurement_codes['code']))) 

tests_results_chems_species_df['measurement_modified'] = tests_results_chems_species_df['measurement'].str.replace('/','')

print(len(set(tests_results_chems_species_df['measurement_modified'])))

print(len(set(tests_results_chems_species_df['measurement_modified']) - set(measurement_codes['code']))) 


In [ ]:
# merging the measurement data

tests_results_chems_species_measuremnt_df = tests_results_chems_species_df.merge(measurement_codes, left_on="measurement_modified", right_on="code", how="inner")
print(tests_results_chems_species_measuremnt_df.shape)
tests_results_chems_species_measuremnt_df.head()


In [ ]:
### Importing Code table

In [ ]:
trend_codes = pd.read_csv("../Data/validation/trend_codes.txt", sep="|", encoding='Windows-1252', dtype=str)
print(trend_codes.shape)
trend_codes.head()

In [ ]:
trend_codes.columns = ["code", "trend_description"]
print(trend_codes.shape)
trend_codes.head()

In [ ]:
tests_results_chems_species_measuremnt_df.columns

In [ ]:
print(set(tests_results_chems_species_measuremnt_df['trend']) - set(trend_codes['code']))
print(set(trend_codes['code']) - set(tests_results_chems_species_measuremnt_df['trend']))

In [ ]:
# merging the trend code

tests_results_chems_species_measuremnt_trend_df = tests_results_chems_species_measuremnt_df.merge(trend_codes, left_on="trend", right_on="code", how="inner")
print(tests_results_chems_species_measuremnt_trend_df.shape)
tests_results_chems_species_measuremnt_trend_df.head()



In [ ]:
### Importing Effect table

In [ ]:
effect_codes = pd.read_csv("../Data/validation/effect_codes.txt", sep="|", encoding='Windows-1252', dtype=str)
print(effect_codes.shape)
effect_codes.head()

In [ ]:
effect_codes.columns = ["code", "effect_description"]
print(effect_codes.shape)
effect_codes.head()

In [ ]:
tests_results_chems_species_measuremnt_trend_df.columns

In [ ]:
print(len(set(tests_results_chems_species_measuremnt_trend_df['effect']) - set(effect_codes['code'])))

tests_results_chems_species_measuremnt_trend_df['effect_modified'] = tests_results_chems_species_measuremnt_trend_df['effect'].str.replace('/','').str.replace('~','')


print(tests_results_chems_species_measuremnt_trend_df.shape)

tests_results_chems_species_measuremnt_trend_df.head()

In [ ]:
# merging effects table
tests_results_chems_species_measuremnt_trend_effect_df = tests_results_chems_species_measuremnt_trend_df.merge(effect_codes, left_on="effect_modified", right_on="code", how="inner")
print(tests_results_chems_species_measuremnt_trend_effect_df.shape)
tests_results_chems_species_measuremnt_trend_effect_df.head()

In [ ]:
### Importing Reference table

In [ ]:
reference_codes = pd.read_csv("../Data/validation/references.txt", sep="|", encoding='Windows-1252', dtype=str)
print(reference_codes.shape)
reference_codes.head()

In [ ]:
tests_results_chems_species_measuremnt_trend_effect_df.columns

In [ ]:
print(set(tests_results_chems_species_measuremnt_trend_effect_df['reference_number']) - set(reference_codes['reference_number']))



In [ ]:
# merging the reference data

tests_results_chems_species_measuremnt_trend_effect_reference_df = tests_results_chems_species_measuremnt_trend_effect_df.merge(reference_codes, on="reference_number", how="inner")
print(tests_results_chems_species_measuremnt_trend_effect_reference_df.shape)
tests_results_chems_species_measuremnt_trend_effect_reference_df.head()

In [ ]:
tests_results_chems_species_measuremnt_trend_effect_reference_df.to_csv("../Output/ecotox_test_results.tsv", sep="\t", index=None, encoding="utf-8")

In [ ]:
print(len(set(tests_results_chems_species_measuremnt_trend_effect_reference_df['cas_number']) - {''}))
print(len(set(tests_results_chems_species_measuremnt_trend_effect_reference_df['measurement_modified']) - {''}))
print(len(set(tests_results_chems_species_measuremnt_trend_effect_reference_df['latin_name']) - {''}))